# Session 5 – What does RAG add? The model alone
NotebookLM answers with **Gemini + your sources** (RAG). This notebook asks **Gemini alone**, with no sources, the same 8 questions.
In class, put each answer next to your NotebookLM answer: whatever differs is what the sources (RAG) added, because the model family is the same.

The instructor runs this notebook before class. Only the questions are sent (Tier 1, public). No PDFs are needed.

In [1]:
import os, sys, re, json, time, getpass, subprocess
try:
    import openai
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "openai"])
from openai import OpenAI

## 1. Connect to Gemini
Use a Gemini API key from Google AI Studio (aistudio.google.com → Get API key). It is not shown or saved.
In Colab you can instead store it as a secret named `GEMINI_API_KEY`.

In [16]:
API_KEY = os.environ.get("GEMINI") or os.environ.get("GOOGLE_API_KEY")
if not API_KEY:
    try:
        from google.colab import userdata
        API_KEY = userdata.get("GEMINI")
    except Exception:
        pass

if not API_KEY:
    raise ValueError("GEMINI API key not found. Please add it to your Colab Secrets (the key icon on the left sidebar) as 'GEMINI'.")

client = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/", api_key=API_KEY, timeout=120, max_retries=0)

MODEL = "gemini-3.1-flash-lite"       # or set by hand from the list below, e.g. MODEL = "gemini-..."
SKIP = ("embed", "image", "tts", "audio", "live", "veo", "imagen", "aqa", "learnlm", "gemma", "robotics", "computer")
ids = sorted({m.id.replace("models/", "") for m in client.models.list().data})
chat_ids = [m for m in ids if m.startswith("gemini") and not any(s in m for s in SKIP)]
print("Gemini chat models on this key:\n  " + "\n  ".join(chat_ids))

def version(m):                                   # gemini-3.1-flash -> (3, 1)
    v = re.search(r"gemini-(\d+)(?:\.(\d+))?", m)
    return (int(v.group(1)), int(v.group(2) or 0)) if v else (0, 0)
stable = [m for m in chat_ids if "flash" in m and "lite" not in m and "preview" not in m and "exp" not in m] or \
         [m for m in chat_ids if "flash" in m and "lite" not in m] or chat_ids
MODEL = MODEL or max(stable, key=lambda m: (version(m), "latest" not in m))
print("\nUsing:", MODEL)

Gemini chat models on this key:
  gemini-2.5-flash
  gemini-2.5-flash-lite
  gemini-2.5-pro
  gemini-3-flash-preview
  gemini-3.1-flash-lite
  gemini-3.1-flash-lite-preview
  gemini-3.1-pro-preview
  gemini-3.1-pro-preview-customtools
  gemini-3.5-flash
  gemini-3.5-flash-lite
  gemini-3.5-transcribe
  gemini-3.6-flash
  gemini-3.7-flash
  gemini-3.8-flash
  gemini-4-argon
  gemini-4-carbon
  gemini-4-hydrogen
  gemini-4-neon
  gemini-decision-1
  gemini-flash-latest
  gemini-flash-lite-latest
  gemini-nano-banana-2.1
  gemini-omni-1.1-flash
  gemini-omni-1.2-flash
  gemini-omni-flash-preview
  gemini-pro-latest

Using: gemini-3.1-flash-lite


In [18]:
SYSTEM = "You are a careful biomedical research assistant. Answer briefly and give your sources."
TIMES = []

def ask(question, system=SYSTEM, max_tokens=4000, tries=4):
    messages = [{"role": "system", "content": system}, {"role": "user", "content": question}]
    for attempt in range(tries):
        t = time.time()
        try:
            r = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.2, max_tokens=max_tokens)
            TIMES.append(time.time() - t)
            return (r.choices[0].message.content or "").strip(), TIMES[-1]
        except Exception as e:
            if "429" in str(e) and attempt < tries - 1:          # free tier: a few requests per minute
                print(f"  rate limit, waiting {20 * (attempt + 1)} s ...")
                time.sleep(20 * (attempt + 1))
            else:
                return f"(no answer: {str(e)[:150]})", time.time() - t

print(ask("Say OK")[0])

OK.


## 2. The 8 questions, model alone
Exactly the worksheet questions. Each answer comes only from what the model learned in training.
The last column of each printout says where the correct answer is, so you can check it in a minute.

In [19]:
QUESTIONS = [
 (1, "Which four factors did Takahashi and Yamanaka use to turn mouse fibroblasts into pluripotent stem cells, and which embryonic stem cell factor turned out not to be needed?",
     "Takahashi & Yamanaka 2006, summary p.1"),
 (2, "Why can a child be born with malformed fingers when no gene's coding sequence is mutated?",
     "Lupiáñez 2015, summary p.1"),
 (3, "What are the seven ways a transcription factor is activated by an outside signal?",
     "Reader p.31 (Fig. 7-34)"),
 (4, "How do enhancers communicate with promoters that are far away along the DNA?",
     "Reader 4.6 + Sabari 2018 abstract"),
 (5, "How does genomic imprinting decide which parent's allele of a gene is expressed?",
     "Not in the sources (0 mentions)"),
 (6, "How do microRNAs regulate gene expression?",
     "Reader p.7 (half a sentence)"),
 (7, "For what fraction of human transcription factors is a DNA-binding motif known?",
     "Lambert 2018 abstract (two-thirds) + p.7 (three-quarters, 1,211)"),
 (8, "Do disruptions of TAD boundaries cause human disease? Give the evidence.",
     "Lupiáñez 2015, summary p.1"),
]
ANSWERS = {}
for n, q, where in QUESTIONS:
    a, t = ask(q)
    ANSWERS[n] = {"question": q, "answer": a, "seconds": round(t, 1), "check": where}
    print(f"{'=' * 90}\nQ{n}  ({t:.0f} s)   check: {where}\n{q}\n{'-' * 90}\n{a}\n")

Q1  (1 s)   check: Takahashi & Yamanaka 2006, summary p.1
Which four factors did Takahashi and Yamanaka use to turn mouse fibroblasts into pluripotent stem cells, and which embryonic stem cell factor turned out not to be needed?
------------------------------------------------------------------------------------------
The four transcription factors used by Takahashi and Yamanaka to reprogram mouse fibroblasts into induced pluripotent stem cells (iPSCs) are **Oct4, Sox2, c-Myc, and Klf4**.

The factor that was initially considered a candidate but proved unnecessary for the reprogramming process is **Nanog**.

**Sources:**
*   Takahashi, K., & Yamanaka, S. (2006). "Induction of pluripotent stem cells from mouse embryonic and adult fibroblast cultures by defined factors." *Cell*, 126(4), 663–676.
*   Yamanaka, S. (2007). "Strategies and new developments in the generation of patient-specific pluripotent stem cells." *Cell Stem Cell*, 1(1), 39–49.

Q2  (2 s)   check: Lupiáñez 2015, summary 

## 3. Run summary and saved answers
The answers are saved to `session5_model_alone.json` for the slides.

In [20]:
if TIMES:
    print(f"Model: {MODEL} | {len(TIMES)} answers, {min(TIMES):.0f}-{max(TIMES):.0f} s each")
else:
    print(f"Model: {MODEL} | 0 answers successfully completed (all timed out or failed)")

json.dump({"model": MODEL, "date": time.strftime("%Y-%m-%d"), "answers": ANSWERS},
          open("session5_model_alone.json", "w"), ensure_ascii=False, indent=1)
print("Saved session5_model_alone.json")

Model: gemini-3.1-flash-lite | 9 answers, 1-2 s each
Saved session5_model_alone.json


## 4. Your own test
Ask the model alone something from this week's reading, then ask NotebookLM the same. What did the sources add?

In [ ]:
print(ask("Your question")[0])